In [ ]:
import pandas as pd

# Replace with the actual path to your CSV file in Drive
file_path = './CLO_SAF_DATA(Sheet1).csv'

df_original = pd.read_csv(file_path , encoding='latin1')
print(df_original.head())

In [ ]:
df_original['id'] = df_original.index + 1
print(df_original.head())

In [40]:
def create_undersampled_negatives_df(arg):
  only_score_one = arg[arg["Score"] == 1]
  undersampled_df = arg[arg['Score'] == 0].sample(n=len(only_score_one), random_state=42)
  new_df = pd.concat([only_score_one, undersampled_df], ignore_index=True)
  new_df = new_df.sample(frac=1, random_state=42).reset_index(drop=True)
  new_df = new_df.copy(deep=True)
  return new_df

In [41]:
from sklearn.model_selection import train_test_split

def create_train_test_eval_df(df):
  train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

  # Second split: eval vs test (50/50 of the 20% temp → 10% each)
  eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

  #print(train_df.head())
  print(len(train_df))
  print(len(train_df[train_df["Score"] == 1]))
  print(len(test_df))
  print(len(eval_df))

  return train_df.copy(deep=True), eval_df.copy(deep=True), test_df.copy(deep=True)

In [42]:
import time
import json
import json5

In [51]:
def process_results(result_df, results, filename):
    test_df = result_df.copy(deep=True)
    test_df['pred_score'] = 0
    test_df['explanation'] = ""
    test_df['classification'] = str([])

    for i in range(len(results)):
        try:
            # Parse the JSON string into a dict if needed
            response = json.loads(results[i]) if isinstance(results[i], str) else results[i]
            
            result_id = response["id"]
            if type(result_id) == str:
                result_id = int(result_id)
            matching_row = test_df[test_df["id"] == result_id]
            if matching_row.empty:
                print(f"Error at index {i} - No matching result_id found in test_df for id: {result_id}")
                continue
            testIdx = test_df.index[test_df["id"] == result_id][0]

            if "explanation" in response:
                test_df.loc[testIdx, 'explanation'] = response["explanation"]
            if "mapLabels" in response:
                test_df.loc[testIdx, 'classification'] = str(response["mapLabels"])
                test_df.loc[testIdx, 'pred_score'] = 1 if len(response["mapLabels"]) > 0 else 0
            else:
                test_df.loc[testIdx, 'classification'] = str([])
                test_df.loc[testIdx, 'pred_score'] = 0
            if "reasoning" in response and len(response["reasoning"]) > 0:
                test_df.loc[testIdx, 'reasoning'] = response["reasoning"]

        except Exception as e:
            print(f"Error at index {i}: {e} \n {results[i]}")

    test_df.to_csv(filename, index=False)
    return test_df

In [44]:
def get_accuracy_result(result_df):
    result_df = result_df.copy(deep=True)
    accuracy = sum(result_df["pred_score"] == result_df["Score"]) / len(result_df)

    New_matching = 0 # original score=0 but predicted label = 1
    Match_for_score_0 = 0
    Match_for_score_1 = 0
    missed_match = 0 #original score = 1 but predicted label = 0

    for i in range(len(result_df)):
        row = result_df.iloc[i]
        if row['Score'] == 1:
            if row['pred_score'] == 1:
                Match_for_score_1 += 1
            else:
                missed_match += 1
        else:
            if row['pred_score'] == 1:
                New_matching += 1
            else:
                Match_for_score_0 += 1
    print(f"Test Accuracy: {accuracy:.3f}")
    print(f"New Matching (original score=0 but predicted label = 1): {New_matching}")
    print(f"Match for score 0: {Match_for_score_0}")
    print(f"Match for score 1: {Match_for_score_1}")
    print(f"Missed Match (original score = 1 but predicted label = 0): {missed_match}")


In [45]:
import ast

def parse_classification_string(classification_str):
    try:
        labels = ast.literal_eval(classification_str)
        flat_list = []
        for item in labels:
            # Only split if item is a string containing ','
            if isinstance(item, str) and ',' in item:
                flat_list.extend([x.strip() for x in item.split(',')])
            else:
                flat_list.append(item)
        filtered_labels = [label.upper() for label in flat_list if label.upper() in {'I', 'R', 'E'}]
        return list(set(filtered_labels))
    except Exception as e:
        print(f"Error parsing classification string: {classification_str} - {e}")
        return list()
        


In [46]:
def process_label_results(df):
    full_match = 0
    partial_match = 0
    complete_mismatch = 0
    newlabelIntroduced = 0

    for i in range(len(df)):
        row = df.iloc[i]
        string_version_pred_labels = parse_classification_string(row['classification'])
        actual_labels = row[['I', 'R', 'E']].tolist()
        clean_labels = [v for v in actual_labels if pd.notna(v)]

        if len(string_version_pred_labels) == 0 and len(clean_labels) != 0:
            complete_mismatch += 1
        elif len(string_version_pred_labels) != 0 and len(clean_labels) == 0:
            complete_mismatch += 1
        elif all(elem in string_version_pred_labels for elem in clean_labels):
            full_match += 1
        elif any(elem in string_version_pred_labels for elem in clean_labels):
            partial_match += 1
        else:
            complete_mismatch += 1

        if any(elem not in clean_labels for elem in string_version_pred_labels):
            newlabelIntroduced += 1


    accuracy= round((full_match+partial_match)/(full_match+partial_match+complete_mismatch), 4)
    new_label_prop = round(newlabelIntroduced/(full_match+partial_match+complete_mismatch), 4)

    print(f"All Labels Match: {full_match}")
    print(f"At Least 1 Label Matches: {partial_match}")
    print(f"No labels Match: {complete_mismatch}")
    print(f"Accuracy with at least 1 correct label prediction: {accuracy} ({round(accuracy*100, 2)}%)")
    print(f"Proportion in which new labels are introduced: {new_label_prop} ({round(new_label_prop*100, 2)}%)")

In [47]:
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import hamming_loss

def calculate_hamming_loss(df):
    true_labels = []
    pred_labels = []
    for i in range(len(df)):
        row = df.iloc[i]
        actual_labels = row[['I', 'R', 'E']].tolist()
        clean_labels = [v for v in actual_labels if pd.notna(v)]
        # print(clean_labels)
        true_labels.append(list(set(clean_labels)))

        filtered_labels = parse_classification_string(row['classification'])
        # print("filtered labels:", filtered_labels, 'original labels:', clean_labels)
        pred_labels.append(filtered_labels)
        # print(string_version_pred_labels)
    mlb = MultiLabelBinarizer()
    mlb.fit(true_labels)
    labels_true_bin = mlb.transform(true_labels)
    labels_pred_bin = mlb.transform(pred_labels)
    print("Label order used:", mlb.classes_)
    # print("True labels:", labels_true_bin)
    # print("Pred Labels:", labels_pred_bin)
    loss = hamming_loss(labels_true_bin, labels_pred_bin)
    print("Hamming loss:", loss)

In [ ]:
def get_list_of_clo__saf(test_df):
    test_df = test_df.copy(deep=True)
    search_string = "Learning Objective"
    matching_col = next((col for col in test_df.columns if search_string.lower() in col.lower()), None)
    test_df = test_df.rename(columns={matching_col: 'CLO'})
    search_string = "SAF"
    matching_col = next((col for col in test_df.columns if search_string.lower() in col.lower()), None)
    test_df = test_df.rename(columns={matching_col: 'Accreditation_Standard'})
    List_of_clo_saf_pairs = test_df[["CLO", "Accreditation_Standard", "id"]].to_dict(orient="records")
    return List_of_clo_saf_pairs

In [ ]:
def build_batch_input(test_df, prompt_template,output_path="sample_batch_input.jsonl"):
    List_of_clo_saf_pairs = get_list_of_clo__saf(test_df)
    print(List_of_clo_saf_pairs)
    with open(output_path, "w", encoding="utf-8") as f:
        for i in range(len(List_of_clo_saf_pairs)):
            pair_text = "\nCLO: " + List_of_clo_saf_pairs[i]['CLO'] + "\nAccreditation_Standard: " + List_of_clo_saf_pairs[i]['Accreditation_Standard'] + "\nid: " + str(List_of_clo_saf_pairs[i]['id']) + "\n<|im_end|>\n <|im_start|>assistant\n/no_think\n"
            record = {"id": str(List_of_clo_saf_pairs[i]['id']), "inputs": prompt_template + pair_text, "parameters": {"max_new_tokens": 1500, "return_full_text": False, "stop": ["<|im_start|>", "<|im_end|>"], "enable_thinking": False}}
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Written {len(test_df)} records to '{output_path}'")


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

ACCESS_KEY = os.getenv("ACCESS_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")
SAGEMAKER_ROLE_ARN = os.getenv("IAM_ROLE_ARN")

In [3]:
import boto3
import json

boto_session = boto3.Session(
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name= "ca-central-1"
)

## Generate input for tests and upload to S3

In [48]:
case1_undersampled_negatives_df = create_undersampled_negatives_df(df_original)
case1_train_df, case1_eval_df, case1_test_df = create_train_test_eval_df(case1_undersampled_negatives_df)

454
223
57
57


In [ ]:
case1_prompt = ''' <|im_start|>system \nYou are an experienced course designer who aligns Course Learning Outcomes (CLOs) with accreditation standards. \nYou follow instructions precisely and always return responses in JSON format for given CLO and accreditation standard pair, the JSON must be the following object (not wrapped in any other key):\n{ \n"id": <given id for CLO and accreditation standard pair>, \n"CLO": "<given CLO>", \n"Accreditation_Standard": "<given accreditation standard>", \n"is_mapped": true or false, \n"mapLabels": ["I", "R", "E"], \n"explanation": "<explanation phrase>", \n} \nNote: A CLO may span multiple mapping scale levels. If so, classify it under all relevant levels. \n<|im_end|>\n<|im_start|>user \n\nAn instructor has developed Course Learning Outcomes (CLOs) for a course syllabus and seeks assistance in aligning each CLO with each accreditation standard and categorizing its cognitive level according to the given mapping scale. \nAs an experienced course designer:\n 1. Compare the given CLO with the given accreditation standard to determine if there is alignment.\n 2. If they align, give an explanation for why they align. \n 3. If they align, classify the CLO into one or more levels of the given mapping scale:\n  Mapping Scale Levels: \n    I — Intoduce \n    R — Reinforce \n    E — Emphasize/Apply \nPlease review the following CLO and accreditation standard and complete the alignment, explanation, and mapping level determination:'''


In [ ]:
build_batch_input(case1_test_df, case1_prompt, output_path="case1_batch_input.jsonl")

In [ ]:
s3 = boto_session.client("s3")

In [ ]:
response = s3.list_buckets()

for bucket in response["Buckets"]:
    print(bucket["Name"])

In [ ]:
def upload_to_s3(s3_client,local_path, bucket_name, s3_key=None):
   
    if s3_key is None:
        s3_key = local_path.split("/")[-1]  # use filename as key if not provided

    s3_client.upload_file(local_path, bucket_name, s3_key)

    s3_uri = f"s3://{bucket_name}/{s3_key}"
    print(f"Uploaded to {s3_uri}")
    return s3_uri


In [ ]:
def check_s3_file(s3, bucket_name, s3_key):
    try:
        response = s3.head_object(Bucket=bucket_name, Key=s3_key)
        print(f"File exists!")
        print(f"Size:          {response['ContentLength'] / 1024:.2f} KB")
        print(f"Last Modified: {response['LastModified']}")
        print(f"ETag:          {response['ETag']}")
    except s3.exceptions.ClientError as e:
        if e.response['Error']['Code'] == "404":
            print("File not found")
        else:
            raise

In [ ]:
input_s3_path = upload_to_s3(s3,
                             "case1_batch_input.jsonl", 
                             "curriculum-map-testing-bucket", 
                             s3_key="input/case1_batch_input.jsonl")

In [ ]:
check_s3_file(s3, "curriculum-map-testing-bucket", "input/case1_batch_input.jsonl")

## Batch Transform with Qwen/Qwen3-8B

In [ ]:
import sagemaker
from sagemaker.huggingface import HuggingFaceModel, get_huggingface_llm_image_uri
from sagemaker.session import Session


sagemaker_session = Session(boto_session=boto_session)

In [5]:
def get_transform_job_logs(job_name, region='ca-central-1'):
    logs_client = boto_session.client('logs', region_name=region)
    
    log_group = '/aws/sagemaker/TransformJobs'
    
    # List all log streams for this job
    streams = logs_client.describe_log_streams(
        logGroupName=log_group,
        logStreamNamePrefix=job_name
    )
    
    for stream in streams['logStreams']:
        print(f"\n--- Stream: {stream['logStreamName']} ---")
        
        response = logs_client.get_log_events(
            logGroupName=log_group,
            logStreamName=stream['logStreamName'],
            startFromHead=True
        )
        
        for event in response['events']:
            print(event['message'])

In [30]:
def parse_results_for_list_response(filename):
    results_think = []
    results = []
    try:
        with open(filename, 'r') as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    result = json.loads(line)
                    if isinstance(result, list):
                        result = result[0] if result else None
                    element = result.get('generated_text') if isinstance(result, dict) else None
                    print(f"Extracted element: {element}")
                    if '</think>' in element:
                        generated_elem_without_think = element.split('</think>')[-1].strip()
                        results.append(generated_elem_without_think)
                        generated_elem_with_think = element.split('</think>')[0].strip()
                        results_think.append(generated_elem_with_think)
                    else:
                        results.append(element)
                except json.JSONDecodeError as e:
                    print(f"Skipping malformed line: {line.strip()} - Error: {e}")
        return results_think, results
    except Exception as e:
        print(f"Error parsing results: {e}")
        return []


In [7]:
MODEL_ID      = "Qwen/Qwen3-8B"
INSTANCE_TYPE = "ml.g5.2xlarge"
S3_INPUT      = "s3://curriculum-map-testing-bucket/input/case1_batch_input.jsonl"     
S3_OUTPUT_FOLDER     = "s3://curriculum-map-testing-bucket/output/"   

In [8]:
hub_env = {
    "HF_MODEL_ID":          MODEL_ID,
    "HF_TASK":              "text-generation",
 
    # Generation defaults — override as needed
    "OPTION_MAX_MODEL_LEN":       "2500",
    "OPTION_MAX_SEQ_LEN":         "2500",
    'SM_NUM_GPUS': json.dumps(1),
    'OPTION_ENABLE_REASONING': 'false'
    
}

In [ ]:
model = HuggingFaceModel(
    image_uri=get_huggingface_llm_image_uri("huggingface",version="3.3.6", region="ca-central-1"),
    env={
        **hub_env,
    },
    role=SAGEMAKER_ROLE_ARN,
    sagemaker_session=sagemaker_session,
    transformers_version="4.51",   
    py_version="py312",
    pytorch_version='2.6.0',
)

In [10]:
transformer = model.transformer(instance_count=1, 
                                instance_type=INSTANCE_TYPE,
                                strategy="SingleRecord",
                                assemble_with="Line",
                                accept="application/json",
                                max_concurrent_transforms=1,
                                output_path="s3://curriculum-map-testing-bucket/output")


In [ ]:
transformer.transform(
    data=S3_INPUT,
    split_type="Line",
    content_type="application/json",
    wait=True,        # blocks until job completes, set False to run async
    logs=True,        # stream logs to console
    job_name="qwen3-8b-batch-transform-huggingface-test-full-input",
    model_client_config={
        "InvocationsTimeoutInSeconds": 120
    }
)

## Clean Up

In [12]:
model.delete_model()

INFO:sagemaker:Deleting model with name: huggingface-pytorch-tgi-inference-2026-03-23-23-01-36-079


In [ ]:
response = sagemaker_session.sagemaker_client.list_endpoint_configs()
print(response)

# sm.close()
# del sm

all_models = sagemaker_session.sagemaker_client.list_models()
print(all_models)

In [ ]:
for model in all_models['Models']:
    model_name = model["ModelName"]
    print(f"Deleting {model_name}...")
    sagemaker_session.sagemaker_client.delete_model(ModelName=model_name)
    
all_models = sagemaker_session.sagemaker_client.list_models()
print(all_models)

In [ ]:
response = sagemaker_session.sagemaker_client.list_transform_jobs()
print(response)

endpoints = sagemaker_session.sagemaker_client.list_endpoints()
print(endpoints)

In [ ]:
# sm = boto3.client("sagemaker", aws_access_key_id=ACCESS_KEY,
#     aws_secret_access_key=SECRET_KEY,region_name="us-west-2")
# response = sm.list_endpoint_configs()
response = sagemaker_session.sagemaker_client.list_endpoint_configs()
print(response)

for config in response['EndpointConfigs']:
   sagemaker_session.sagemaker_client.delete_endpoint_config(EndpointConfigName=config['EndpointConfigName'])
  
response = sagemaker_session.sagemaker_client.list_endpoint_configs()
print(response)

# sm.close()
# del sm

all_models = sagemaker_session.sagemaker_client.list_models()
print(all_models)

for model_info in all_models['Models']:
   model_name = model_info['ModelName']
   sagemaker_session.sagemaker_client.delete_model(ModelName=model_name)
  
check_models = sagemaker_session.sagemaker_client.list_models()
print(check_models)

all_model_cards = sagemaker_session.sagemaker_client.list_model_cards()
print(all_model_cards)

all_inference_components = sagemaker_session.sagemaker_client.list_inference_components()
print(all_inference_components)

all_images = sagemaker_session.sagemaker_client.list_images()
print(all_images)

all_artifacts = sagemaker_session.sagemaker_client.list_artifacts()
print(all_artifacts)

all_labelling_jobs = sagemaker_session.sagemaker_client.list_labeling_jobs()
print(all_labelling_jobs)

all_notebooks = sagemaker_session.sagemaker_client.list_notebook_instances()
print(all_notebooks)

all_domains = sagemaker_session.sagemaker_client.list_domains()
print(all_domains)


In [ ]:
all_artifacts = sagemaker_session.sagemaker_client.list_artifacts()
print(all_artifacts)

for artifact in all_artifacts['ArtifactSummaries']:
    
    # First delete all associations
    associations = sagemaker_session.sagemaker_client.list_associations(SourceArn=artifact['ArtifactArn'])
    for assoc in associations['AssociationSummaries']:
        sagemaker_session.sagemaker_client.delete_association(
            SourceArn=assoc['SourceArn'],
            DestinationArn=assoc['DestinationArn']
        )
        print(f"Deleted association: {assoc['SourceArn']} -> {assoc['DestinationArn']}")
    
    # Also check if artifact is a destination in other associations
    associations = sagemaker_session.sagemaker_client.list_associations(DestinationArn=artifact['ArtifactArn'])
    for assoc in associations['AssociationSummaries']:
        sagemaker_session.sagemaker_client.delete_association(
            SourceArn=assoc['SourceArn'],
            DestinationArn=assoc['DestinationArn']
        )
        print(f"Deleted association: {assoc['SourceArn']} -> {assoc['DestinationArn']}")
    sagemaker_session.sagemaker_client.delete_artifact(ArtifactArn=artifact['ArtifactArn'])
    print(f"Deleted artifact: {artifact['ArtifactArn']}")

In [ ]:
#get_transform_job_logs('qwen3-8b-batch-transform-huggingface-test')

## Get Output from S3

In [18]:
job = sagemaker_session.sagemaker_client.describe_transform_job(TransformJobName='qwen3-8b-batch-transform-huggingface-test-full-input')
output_s3_uri = job['TransformOutput']['S3OutputPath']
print("Output location:", output_s3_uri)

Output location: s3://curriculum-map-testing-bucket/output


In [24]:
bucket = output_s3_uri.replace('s3://', '').split('/')[0]
prefix = '/'.join(output_s3_uri.replace('s3://', '').split('/')[1:])

# Download all output files
s3 = boto_session.client('s3')

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

In [25]:
print(bucket, prefix)

curriculum-map-testing-bucket output


In [ ]:
for obj in response['Contents']:
    key = obj['Key']
    filename = key.split('/')[-1]
    print(f"Downloading {filename}...")
    s3.download_file(bucket, key, filename)
    print(f"Saved to ./{filename}")

In [ ]:
parsed_results_think, parsed_results = parse_results_for_list_response('case1_batch_input.jsonl.out')

In [77]:
print(len(parsed_results_think), len(parsed_results))

57 57


## Case 1: Results with JSON objects before think 

In [78]:
case1_df_with_predictions = process_results(case1_test_df, parsed_results_think, './TestBatchTransform(Qwen3-8B-Before-think).csv')

Error at index 11: Expecting value: line 1 column 1 (char 0) 
 
Error at index 22: Expecting value: line 1 column 1 (char 0) 
 
Error at index 23: Expecting value: line 1 column 1 (char 0) 
 
Error at index 30: Expecting value: line 1 column 1 (char 0) 
 
Error at index 43: Expecting value: line 1 column 1 (char 0) 
 
Error at index 54: Expecting value: line 1 column 1 (char 0) 
 


In [79]:
get_accuracy_result(case1_df_with_predictions)

Test Accuracy: 0.807
New Matching (original score=0 but predicted label = 1): 4
Match for score 0: 22
Match for score 1: 24
Missed Match (original score = 1 but predicted label = 0): 7


In [80]:
process_label_results(case1_df_with_predictions)

All Labels Match: 37
At Least 1 Label Matches: 5
No labels Match: 15
Accuracy with at least 1 correct label prediction: 0.7368 (73.68%)
Proportion in which new labels are introduced: 0.2632 (26.32%)


In [81]:
calculate_hamming_loss(case1_df_with_predictions)

Label order used: ['E' 'I' 'R']
Hamming loss: 0.2631578947368421


## Case 2: Tests with JSON objects after think

In [82]:
case2_undersampled_negatives_df = create_undersampled_negatives_df(df_original)
case2_train_df, case2_eval_df, case2_test_df = create_train_test_eval_df(case2_undersampled_negatives_df)

454
223
57
57


In [83]:
case2_df_with_predictions = process_results(case2_test_df, parsed_results, './TestBatchTransform(Qwen3-8B-After-think).csv')

In [84]:
get_accuracy_result(case2_df_with_predictions)

Test Accuracy: 0.877
New Matching (original score=0 but predicted label = 1): 5
Match for score 0: 21
Match for score 1: 29
Missed Match (original score = 1 but predicted label = 0): 2


In [85]:
process_label_results(case2_df_with_predictions)

All Labels Match: 43
At Least 1 Label Matches: 4
No labels Match: 10
Accuracy with at least 1 correct label prediction: 0.8246 (82.46%)
Proportion in which new labels are introduced: 0.3509 (35.09%)


In [86]:
calculate_hamming_loss(case2_df_with_predictions)

Label order used: ['E' 'I' 'R']
Hamming loss: 0.22807017543859648
